# LoRA Fine-tuning for Language Learning Chatbot

This notebook fine-tunes Qwen2.5-7B-Instruct using LoRA (Low-Rank Adaptation) for a language learning chatbot.

In [ ]:
from huggingface_hub import login, create_repo, upload_folder
from kaggle_secrets import UserSecretsClient
from accelerate import notebook_launcher

user_secrets = UserSecretsClient()
HF_TOKEN_W = user_secrets.get_secret("HF_TOKEN_W")
login(HF_TOKEN_W)

In [ ]:
def train_fn():
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig, TrainerCallback
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer
    from accelerate import Accelerator
    from datasets import load_dataset
    import torch
    import math
    import json
    
    accelerator = Accelerator()
    
    # Re-define constants to avoid pickling issues
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    language = "Korean"
    data_path = f'/kaggle/input/chatbot-dataset/{language}_training_data.jsonl'
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load and prepare dataset
    ds = load_dataset("json", data_files=data_path, split="train")
    
    def format_chat(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
        return {"text": text}
    
    ds = ds.map(format_chat, remove_columns=ds.column_names)
    ds = ds.train_test_split(test_size=0.05, seed=42)
    
    # Metrics callback (only collect on main process)
    class MetricsCallback(TrainerCallback):
        def __init__(self):
            self.train_losses = []
            self.eval_losses = []
            self.perplexities = []
            self.steps = []

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is None:
                return
            if "loss" in logs:
                self.train_losses.append(logs["loss"])
                self.steps.append(state.global_step)

        def on_evaluate(self, args, state, control, metrics=None, **kwargs):
            if metrics is None:
                return
            eval_loss = metrics.get("eval_loss")
            if eval_loss is not None:
                self.eval_losses.append(eval_loss)
                self.perplexities.append(math.exp(eval_loss))
    
    metrics_callback = MetricsCallback()
    
    # Load model on the correct GPU for this process
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    base = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map={"": accelerator.local_process_index}
    )
    base.gradient_checkpointing_enable()
    base.config.use_cache = False
    
    # Apply LoRA
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    
    model = get_peft_model(base, lora_config)
    
    if accelerator.is_main_process:
        model.print_trainable_parameters()
    
    # Training arguments
    training_args = TrainingArguments(
        output_dir=f"/kaggle/working/qwen/{language}/checkpoints",

        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,

        learning_rate=1e-4,
        num_train_epochs=5,

        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        optim="paged_adamw_8bit",

        logging_steps=50,

        eval_strategy="steps",
        eval_steps=100,

        save_steps=100,
        save_total_limit=3,

        ddp_find_unused_parameters=False,

        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        
        report_to="none"
    )
    
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=ds["train"],
        eval_dataset=ds["test"],
        args=training_args,
        callbacks=[metrics_callback],
        max_seq_length=1024,
        packing=True,
    )
    
    trainer.train()
    
    # Save only from main process
    if accelerator.is_main_process:
        final_dir = f"/kaggle/working/qwen/{language}/final"
        trainer.model.save_pretrained(final_dir)
        tokenizer.save_pretrained(final_dir)
        
        # Save metrics for plotting
        metrics_data = {
            "train_losses": metrics_callback.train_losses,
            "eval_losses": metrics_callback.eval_losses,
            "perplexities": metrics_callback.perplexities,
            "steps": metrics_callback.steps
        }
        with open(f"/kaggle/working/qwen/{language}/metrics.json", "w") as f:
            json.dump(metrics_data, f)

In [ ]:
# Launch training on both GPUs
notebook_launcher(train_fn, num_processes=2)

In [ ]:
import json
import matplotlib.pyplot as plt

LANGUAGE = "Korean"
OUTPUT_DIR = f"/kaggle/working/qwen/{LANGUAGE}"

with open(f"{OUTPUT_DIR}/metrics.json", "r") as f:
    metrics = json.load(f)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(metrics["steps"], metrics["train_losses"])
axes[0].set_xlabel("Steps")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")

axes[1].plot(metrics["eval_losses"])
axes[1].set_xlabel("Eval Step")
axes[1].set_ylabel("Loss")
axes[1].set_title("Validation Loss")

axes[2].plot(metrics["perplexities"])
axes[2].set_xlabel("Eval Step")
axes[2].set_ylabel("Perplexity")
axes[2].set_title("Perplexity")

plt.tight_layout()
plt.show()

In [ ]:
api = HfApi(token=HF_TOKEN_W)


model_name = "Qwen/Qwen2.5-7B-Instruct"
model_name_for_repo = model_name.split('/')[-1]
OUTPUT_REPO = f"jiminaa/{model_name_for_repo}-language-QLoRA"

create_repo(repo_id=OUTPUT_REPO, exist_ok=True, repo_type="model", token=HF_TOKEN_W)

api.upload_folder(
    folder_path=f"{OUTPUT_DIR}/final",
    repo_id=OUTPUT_REPO,
    path_in_repo=f"{LANGUAGE}",
    commit_message=f"Upload {LANGUAGE} LoRA adapters"
)